In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error
)

clf_df = pd.read_csv("classification_escalation.csv")
reg_df = pd.read_csv("resolution_time_regression.csv")

display(clf_df.head())
display(reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [3]:
X = clf_df.drop(columns=["ticket_id", "escalated"])
y = clf_df["escalated"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

classification_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

classification_results = []

for name, model in classification_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=classification_cv,
        scoring="f1"
    )

    classification_results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    })

classification_results_df = pd.DataFrame(classification_results)

display(classification_results_df)

,Model,Mean F1,Std F1
0,Logistic Regression,0.392651,0.062747
1,K-Nearest Neighbors,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


In [5]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

classification_search = GridSearchCV(
    logistic_pipeline,
    param_grid={
        "model__C": [0.01, 0.1, 1, 10, 100]
    },
    cv=classification_cv,
    scoring="f1",
    n_jobs=1
)

classification_search.fit(X_train, y_train)

print("Best Parameters:", classification_search.best_params_)
print("Best Cross-Validation F1:", classification_search.best_score_)

y_pred = classification_search.predict(X_test)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Best Parameters: {'model__C': 10}
Best Cross-Validation F1: 0.4225932078257659

Confusion Matrix:
[[137   5]
 [ 30   8]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180



In [6]:
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
y_reg = reg_df["resolution_time_hours"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42
)

numeric_features_reg = X_reg.select_dtypes(include=np.number).columns.tolist()
categorical_features_reg = X_reg.select_dtypes(exclude=np.number).columns.tolist()

numeric_preprocessor_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_reg = ColumnTransformer([
    ("numeric", numeric_preprocessor_reg, numeric_features_reg),
    ("categorical", categorical_preprocessor_reg, categorical_features_reg)
])

regression_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
regression_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42)
}

regression_results = []

for name, model in regression_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor_reg),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train_reg,
        y_train_reg,
        cv=regression_cv,
        scoring="neg_mean_absolute_error"
    )

    mae_scores = -scores["test_score"]

    regression_results.append({
        "Model": name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

regression_results_df = pd.DataFrame(regression_results)

display(regression_results_df)

,Model,Mean MAE,Std MAE
0,Linear Regression,2.035134,0.171650
1,Ridge Regression,2.035127,0.171404
2,Decision Tree Regressor,3.239097,0.053807


In [8]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", Ridge())
])

ridge_search = GridSearchCV(
    ridge_pipeline,
    param_grid={
        "model__alpha": [0.01, 0.1, 1, 10, 100]
    },
    cv=regression_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=1
)

ridge_search.fit(X_train_reg, y_train_reg)

print("Best Parameters:", ridge_search.best_params_)
print("Best Cross-Validation MAE:", -ridge_search.best_score_)

reg_pred = ridge_search.predict(X_test_reg)

test_mae = mean_absolute_error(y_test_reg, reg_pred)

print("Test MAE:", test_mae)

Best Parameters: {'model__alpha': 0.1}
Best Cross-Validation MAE: 2.035118085555478
Test MAE: 2.230086144180656


Required report 
Your final report should include a problem framing(what are you solving or predicting), 
what are you doing during preprocessing and why. Explanation and when applicable, 
justification for: validation strategy, candidate models and search space, CV results table, 
final test result,  leakage prevention,  final model justification and limitations. 


Experiment A — Classification_escalation

1, Problem framing
The aim is to create a classification model, which will predict if a support ticket will be escalated or not.
The target variable is escalated and the task is a binary classification:
- 0: Ticket is not escalated:
-1: Escalates the ticket.
The model makes predictions based on ticket data like complexity score, team experience, priority, channel and customer tenure.

2. Preprocessing and its importance
This was done by applying the following preprocessing steps:
Check the common mistakes listed below when writing a ticket:When writing a ticket, look for the common mistakes below and avoid them:
Missing-value imputation: Values that are missing for numerical variables are filled in with the value of the median, and missing values for categorical variables with the most common category.
Standardization (numerical): Numerical features are scaled using Standard Scaler so that features with larger numerical ranges will not dominate distance-based or regularized models.
One-hot encoding: Categorical features were transformed to numerical columns which machine-learning models could use.
- Unknown categories: handle_unknown="ignore" means that the model can process inputs that did not contain the categories during the training.

3. Validation strategy
The split was 80/20, and random_state=42 was employed to ensure the split was reproducible.
The split into classes is done via stratification since the classes are imbalanced. Of all the tickets, 78.78% are not escalated, and 21.22% are escalated.Five-fold Stratified Cross-Validation was used to compare the models. Both the number of escalated and non-escalated tickets remain fairly consistent in each fold. For a less abundant class, the F1 score was chosen as the main score as it takes both precision and recall into account.

4. Candidate models
So three classification algorithms were compared:
- Logistic Regression: An easy-to-interpret classification algorithm with a decent baseline result.
- K-Nearest Neighbors (KNN): Classifies a ticket by the classes of the training examples that are nearest to it.
- Decision Tree: It can capture the nonlinear relationship, and can learn decision rules.
To make the comparison fair all three models are given the same preprocessing pipeline and cross validation strategy.

5. Cross-validation results
Five-fold cross-validation — F1 score, higher is better.
Model	                    Mean F1	        Standard deviation
Logistic Regression      	      0.3927	    0.0627
KNN                            	  0.3608	    0.0503
Decision Tree	                  0.3307	    0.0581

Logistic Regression achieved the highest mean F1 score. Therefore, it was selected for hyperparameter tuning.

6. Hyperparameter tuning
GridSearchCV was used to tune the regularization parameter C of Logistic Regression.

The search space was:
[0.01, 0.1, 1, 10, 100]

The best parameter was:
- C = 10
- Best cross-validation F1 score: approximately 0.4226
The parameter C controls the strength of regularization. A smaller value applies stronger regularization, while a larger value applies weaker regularization.

7. Final test result
The selected Logistic Regression pipeline was evaluated on the test set.

Confusion matrix:
[[137   5]
 [ 30   8]]

Classification report:
Class	                  Precision	          Recall	     F1-score
Not escalated	             0.82	           0.96	           0.89
Escalated	                 0.62	           0.21	           0.31

- Accuracy: 0.81
- False positives: 5
- False negatives: 30
The model correctly identifies most tickets that are not escalated. However, it misses many tickets that actually become escalated.

This is an important limitation because missing an escalation may be more costly than incorrectly predicting one.

8. Leakage prevention
Data leakage was prevented through the following steps:
- The test set is separated prior to training the model and cross validation.
- Imputation of missing values was inserted within the preprocessing pipeline.
- Scaling is added within the pre-processing pipeline.
- One-hot encoding also was used within the preprocessing pipeline.
- Preprocessing is only fitted on the training set of each of the cross-validation folds.
- The test set was neither used for model selection nor for the tuning of hyperparameters.
This allows information from the validation or test data to not affect preprocessing during training.

9. Final model justification and limitations
Logistic Regression was chosen as the best mean F1 score of the three initial classification candidates.
But, its test recall for escalated tickets is just 0.21. This translates to it only recognising a small fraction of real escalations.
The model can be used as a first pass tool, but should not be used as a sole ticket identification tool.
Potential enhancements include modifying the classification threshold, conducting testing with class weighted models, and comparing the costs of false negatives with false positives.


Experiment B — resolution_time_regression

1. Problem framing
The goal is to estimate the amount of hours it will take to resolve a support ticket.
The variable to be predicted is resolution_time_hours.
It is a regression problem as it predicts a continuous numerical value, not a category.

2. Preprocessing and why it is needed
The subsequent pre-processing steps were carried out:
- Remove ticket_id: An identifier and is not helpful in predicting resolution time.
- Imputation of Missing Values: Numerical missing values are filled in with the median value, while categorical missing values are imputed with the --most frequently occurring category.
- Standardization: Numerical features are scaled to give ranges that are similar, especially for Ridge Regression.
- Categorical features are converted to numerical representations: One-hot encoding.
- When you have some categories that are unknown, do not want it to cause errors, and use handle_unknown="ignore".

3. Validation strategy
The train-test split was done as an 80/20 split with random state=4
Five-fold KFold Cross-Validation was chosen since this is a regression problem with a continuous target variable.

The folds are shuffled prior to splitting, which helps to minimize dependence on the original data order.

The main evaluation metric chosen was Mean Absolute Error (MAE), which calculates the mean absolute error between the true and the predicted resolution times.

Better predictions are given by a lower MAE.

4. Candidate models
Three algorithms for regression were compared:
- Linear Regression: Fits a simple baseline, using linear relationship between features and resolution time.
- Ridge Regression: An extension of Linear Regression with regularization to keep the influence of large coefficients in check and to prevent overfitting.
- Decision Tree Regressor: Captures non-linear relationships by partitioning the data into decision-region regions.

5. Cross-validation results
Five-fold cross-validation — MAE in hours, lower is better.
Model	                        Mean MAE	   Standard deviation
Linear Regression              	 2.0351      	0.1717
Ridge Regression	             2.0351     	0.1714
Decision Tree Regressor	         3.2692	        0.1439

Linear Regression and Ridge Regression achieved almost identical results. Both performed better than the Decision Tree Regressor.
Ridge Regression was selected for hyperparameter tuning because it achieved the lowest mean MAE, although its advantage over Linear Regression was extremely small.

6. Hyperparameter tuning
GridSearchCV was used to tune the Ridge Regression parameter alpha.

The search space was:
[0.01, 0.1, 1, 10, 100]
The best parameter was:
- alpha = 0.1
- Best cross-validation MAE: approximately 2.0351 hours
The alpha parameter controls the strength of regularization. Increasing it applies stronger regularization to the model coefficients.

7. Final test result
The selected Ridge Regression pipeline was evaluated on the test set.
Final test MAE: 2.23 hours
This means that the model's predictions differ from the actual resolution times by approximately 2.23 hours on average.

8. Leakage prevention
To prevent any data leakage, the following steps were taken:
- Splitting the test set prior selection of the model.
- Including imputation, scaling and encoding within the pre-processing pipeline.
- Only preprocessing during training phase on the training folds of the cross validation.
- Hyperparameter Tuning based on the training set only.
- Evaluating the test set after the final model has been selected.

9. Model Explanation and Assumptions
Ridge Regression was chosen as it had the lowest mean cross-validation MAE and it also offers regularization that helps to control the complexity of the model.
But its performance was very similar to regular Linear Regression. The improvement is thus very small.
The regression target also has quite low values and is skewed to the right. There are 72 observations at exactly 0.5 hours, and a few tickets take a lot longer to be resolved.
The extreme values may have an impact on the regression.
The MAE is less sensitive to large errors than is Root Mean Squared Error (RMSE), which squares the prediction errors before taking the mean. Thus, RMSE emphasizes larger prediction errors more.
The model might not be able to predict resolution time that is different from the norm and, if applied to an actual support system, its efficacy should be monitored.

Also include 
reflections of the following questions: 

1. Why stratify the classification split? 
The number of each target class in the train/validation sets is roughly the same.
This is crucial when the data set is imbalanced. If there is no stratification, then the test set may be too small for the evaluation to be reliable.

2. Why must preprocessing be inside the pipeline? 
To avoid data leakage, pre-processing should be done within the pipeline.
For instance, when scaling or imputation is done prior to cross-validation, the information from the validation folds can be used in scaling or imputation.
A pipeline ensures the preprocessing will be applied only to the training part of each fold, and then to the validation part.

3. What does fold variability tell you? 
The variability of fold represents the difference between the performance of different folds in cross validation.
Small standard deviation indicates that the model is relatively consistent across folds.
High standard deviation is indicative of that the model performance varies significantly across folds.
It assists us in comprehending the stableness and reliability of a model.

4. Why should the test set be evaluated only once? 
The test set will give an unbiased final test of the selected model.
Using test information indirectly during model selection when we repeatedly evaluate the test set and then use the results of that evaluation to alter the model.
This can result in overfitting on test set which may produce overly optimistic estimates.

5. What deployment constraints might change your choice? 
There are a number of practical considerations that could affect the actual decision on which model to use in practice:
Prediction speed: When predictions need to be made rapidly, the simpler the better.
- Interpretability: Logistic Regression and Linear Regression can be preferred if users wish to know the predictions.
- Computational resources: Models that use less memory or less computation may be more appropriate.
- Accuracy needs: If the loss of an escalated ticket is expensive, a model should have improved recall.
- Maintenance: An easy-to-maintain and monitor model may be better.
- Changing data: If patterns in tickets are changing over time, model performance should be expected to drop.
Not all the best models are the ones with the highest evaluation score. It also needs to meet the operational needs of the system.


Conclusion

The lab tested three classifiers and three regressors with preprocessing pipelines and 5-fold cross validation.

Logistic Regression had the highest mean F1 score and was chosen for hyperparameter tuning for classification. Final results of its tests were favorable for detection of non-escalated tickets and poor for the detection of escalated tickets.

Ridge Regression had the lowest mean CV MAE for regression but was nearly as good as Linear Regression. The last test MAE was around 2.23 hours.

Overall, the experiments demonstrate the importance of proper preprocessing, cross-validation, hyperparameter tuning, and leakage prevention when developing reliable machine-learning models.
